# 🔥 hCaptcha Demo Scraper → Vision Trainer → GGUF

**One notebook, end to end.** No Discord bans, no proxies needed.

```
hcaptcha.com/demo → type "1" → submit → hCaptcha appears
  → screenshot + extract tiles → refresh → repeat 100x
  → build labeled dataset → train Qwen2.5-VL → export GGUF
```

In [ ]:
# ═══════════════════════════════════════
# 1. KILL THE RUNTIME & RESTART FIRST
# ═══════════════════════════════════════
# Menu → Runtime → Restart and run all
# This avoids the PicklingError from stale trl versions

!pip install -q transformers==4.51.3 trl==0.21.0 huggingface-hub>=1.2.0
!pip install -q unsloth unsloth-zoo datasets accelerate bitsandbytes peft
!pip install -q nodriver Pillow

import asyncio, json, os, random, re, time, base64, gc
from pathlib import Path
from PIL import Image
import torch

DATA = Path("/content/hcaptcha_data")
DATA.mkdir(exist_ok=True)
(DATA / "challenges").mkdir(exist_ok=True)
(DATA / "tiles").mkdir(exist_ok=True)
(DATA / "synthetic").mkdir(exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print("✅ Ready")

In [ ]:
# ═══════════════════════════════════════
# 2. SCRAPER: hCaptcha Demo Loop
# ═══════════════════════════════════════
# Goes to accounts.hcaptcha.com/demo, fills field,
# submits, waits for hCaptcha, screenshots it,
# then refreshes and does it again.

import nodriver as uc

HCAPTCHA_DEMO = "https://accounts.hcaptcha.com/demo"

class DemoScraper:
    """Scrape the OFFICIAL hCaptcha demo page. No bans, no proxies."""

    collected = []

    async def scrape_one(self) -> dict | None:
        browser = await uc.start(
            browser_args=["--no-sandbox", "--disable-dev-shm-usage",
                          "--disable-blink-features=AutomationControlled",
                          "--window-size=1400,900"],
            headless=True, sandbox=False
        )
        try:
            page = await browser.get(HCAPTCHA_DEMO)
            await asyncio.sleep(3)

            # Find the input field (demo page has a textarea/input)
            inp = await page.find("textarea, input[type='text']", timeout=5)
            if inp:
                await inp.click()
                await asyncio.sleep(0.3)
                await inp.send_keys("1")

            # Find and click submit button
            btn = await page.find("button[type='submit'], input[type='submit']", timeout=3)
            if not btn:
                btn = await page.find("button", timeout=3)
            if btn:
                await btn.click()

            # Wait for hCaptcha iframe to render
            await asyncio.sleep(4)

            html = await page.get_content()

            if "hcaptcha" in html.lower():
                # Check if the challenge iframe appeared
                has_challenge = "checkbox" not in html.lower()[:2000] or "challenge" in html.lower()

                screenshot = await page.screenshot()
                ts = int(time.time() * 1000)
                fname = DATA / "challenges" / f"{ts}.png"
                with open(fname, "wb") as f:
                    f.write(screenshot)

                result = {
                    "screenshot": str(fname),
                    "timestamp": ts,
                    "has_challenge": has_challenge
                }
                print(f"  ✅ Saved: {fname.name}")
                return result
            else:
                print("  ⚠️ No hCaptcha found")
                return None

        except Exception as e:
            print(f"  ❌ {e}")
            return None
        finally:
            try: await browser.stop()
            except: pass

    async def scrape_loop(self, count=100, delay=3.0):
        print(f"\n🎯 Scraping {count} hCaptcha challenges...\n")
        results = []
        for i in range(count):
            print(f"[{i+1}/{count}]", end=" ")
            r = await self.scrape_one()
            if r: results.append(r)
            await asyncio.sleep(delay + random.uniform(0, 2))

        with open(DATA / "challenges" / "index.json", "w") as f:
            json.dump(results, f, indent=2)

        print(f"\n✅ {len(results)}/{count} captured")
        return results

scraper = DemoScraper()
print("✅ Scraper ready — targets hcaptcha.com/demo")

In [ ]:
# ═══════════════════════════════════════
# 3. RUN THE SCRAPER
# ═══════════════════════════════════════
# This opens a stealth Chrome, types "1",
# submits, screenshots the hCaptcha, refreshes.
# 100 loops × ~7s each ≈ 12 minutes.

scraped = await scraper.scrape_loop(count=100, delay=5.0)

In [ ]:
# ═══════════════════════════════════════
# 4. EXTRACT TILES FROM SCREENSHOTS
# ═══════════════════════════════════════
# hCaptcha shows a 3x3 grid of images.
# We crop each tile out as a 128x128 image.
# The prompt text tells us what to look for.

from PIL import Image, ImageFilter

def extract_grid_from_screenshot(path: str) -> list[Image.Image]:
    """
    Crop the 3x3 hCaptcha grid from a full-page screenshot.
    Grid is roughly centered in the iframe area.
    Returns 9 tiles.
    """
    img = Image.open(path).convert("RGB")
    w, h = img.size

    # hCaptcha grid is roughly 400x400 pixels in a ~1400x900 viewport.
    # It's in the center-left area of the page (the iframe isn't fullscreen).
    # We try to find it by looking for a roughly square region with content.

    # Approximate grid location (centered in the iframe, which is on the left side)
    grid_w, grid_h = 360, 360
    left = (w - grid_w) // 2 - 100   # offset left for the form on the right
    top = (h - grid_h) // 2

    # Clamp to image bounds
    left = max(0, min(left, w - grid_w))
    top = max(0, min(top, h - grid_h))

    tile_size = grid_w // 3
    tiles = []
    for row in range(3):
        for col in range(3):
            x = left + col * tile_size
            y = top + row * tile_size
            tile = img.crop((x, y, x + tile_size, y + tile_size))
            tile = tile.resize((128, 128), Image.LANCZOS)
            tiles.append(tile)
    return tiles


def process_all_challenges():
    """Process all scraped screenshots into tiles"""
    challenge_dir = DATA / "challenges"
    all_tiles = []

    pngs = sorted(challenge_dir.glob("*.png"))
    print(f"Processing {len(pngs)} screenshots...")

    for png in pngs:
        try:
            tiles = extract_grid_from_screenshot(str(png))
            for j, tile in enumerate(tiles):
                tile_name = f"{png.stem}_{j}.png"
                tile_path = DATA / "tiles" / tile_name
                tile.save(tile_path)
                all_tiles.append(str(tile_path))
        except Exception as e:
            print(f"  ⚠️ Failed {png.name}: {e}")

    print(f"✅ Extracted {len(all_tiles)} tiles from {len(pngs)} screenshots")
    return all_tiles

tiles = process_all_challenges()

In [ ]:
# ═══════════════════════════════════════
# 5. SYNTHETIC DATASET (labeled!)
# ═══════════════════════════════════════
# Since we can't auto-label real hCaptcha tiles
# (we'd need the answer key), we generate synthetic
# tiles that mimic hCaptcha's style.
#
# This gives us 10k+ perfectly labeled examples.
# The real scraped tiles go into the eval set.

import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance

class HCaptchaSynth:
    """Generate synthetic hCaptcha-style 3x3 challenge tiles"""

    OBJECTS = [
        "bus", "car", "bicycle", "motorcycle", "truck", "boat", "airplane",
        "train", "traffic light", "fire hydrant", "stop sign",
        "crosswalk", "bridge", "stairs", "chimney", "cat", "dog", "bird",
        "palm tree", "bicycle"
    ]

    def __init__(self, output_dir: Path):
        self.out = output_dir
        self.out.mkdir(exist_ok=True)

    def _bg(self, draw, img):
        """Add noisy abstract background"""
        w = 128
        # Random noise
        for _ in range(80):
            x, y = random.randint(0,w), random.randint(0,w)
            r, g, b = random.randint(30,230), random.randint(30,230), random.randint(30,230)
            draw.ellipse([x-2,y-2,x+2,y+2], fill=(r,g,b))
        # Random lines
        for _ in range(random.randint(2,6)):
            draw.line([random.randint(0,w), random.randint(0,w),
                       random.randint(0,w), random.randint(0,w)],
                      fill=(random.randint(50,200),)*3, width=random.randint(1,3))

    def _draw(self, draw, obj, cx=64, cy=64):
        """Draw a rough icon for the object"""
        if obj == "bus":
            draw.rectangle([cx-35,cy-18,cx+35,cy+12], fill=(220,180,30), outline=(0,0,0))
            draw.ellipse([cx-24,cy+8,cx-14,cy+18], fill=(30,)*3)
            draw.ellipse([cx+14,cy+8,cx+24,cy+18], fill=(30,)*3)
            draw.rectangle([cx-24,cy-15,cx-10,cy-8], fill=(150,200,255))
            draw.rectangle([cx,cy-15,cx+14,cy-8], fill=(150,200,255))
        elif obj == "car":
            pts = [cx-25,cy+8, cx-18,cy-10, cx+12,cy-10, cx+22,cy+8]
            draw.polygon(pts, fill=(200,50,50), outline=(0,0,0))
            draw.ellipse([cx-14,cy+6,cx-4,cy+16], fill=(30,)*3)
            draw.ellipse([cx+4,cy+6,cx+14,cy+16], fill=(30,)*3)
        elif obj in ("bicycle",):
            draw.ellipse([cx-18,cy-8,cx-2,cy+14], outline=(80,)*3, width=2)
            draw.ellipse([cx+2,cy-8,cx+18,cy+14], outline=(80,)*3, width=2)
            draw.line([cx,cy+3,cx,cy-8], fill=(80,)*3, width=2)
            draw.line([cx,cy-3,cx-10,cy-10], fill=(80,)*3, width=2)
        elif obj == "motorcycle":
            draw.ellipse([cx-14,cy+2,cx,cy+16], fill=(30,)*3)
            draw.ellipse([cx,cy+2,cx+14,cy+16], fill=(30,)*3)
            draw.rectangle([cx-10,cy-10,cx+10,cy+2], fill=(200,100,30))
            draw.line([cx,cy-10,cx,cy-20], fill=(150,)*3, width=2)
        elif obj == "truck":
            draw.rectangle([cx-35,cy-15,cx+20,cy+10], fill=(100,150,200), outline=(0,0,0))
            draw.rectangle([cx+20,cy-5,cx+35,cy+10], fill=(80,120,160), outline=(0,0,0))
            draw.ellipse([cx-25,cy+6,cx-15,cy+16], fill=(30,)*3)
            draw.ellipse([cx+10,cy+6,cx+20,cy+16], fill=(30,)*3)
        elif obj == "boat":
            draw.polygon([cx-25,cy+5,cx+25,cy+5,cx+15,cy+15,cx-15,cy+15], fill=(150,100,50))
            draw.polygon([cx-5,cy-10,cx,cy-20,cx+10,cy-5], fill=(255,255,255))
        elif obj == "airplane":
            draw.ellipse([cx-20,cy-5,cx+20,cy+5], fill=(200,)*3, outline=(0,0,0))
            draw.polygon([cx-20,cy-5,cx-35,cy-15,cx-30,cy], fill=(200,)*3)
            draw.polygon([cx+20,cy-5,cx+35,cy-15,cx+30,cy], fill=(200,)*3)
            draw.polygon([cx-5,cy-15,cx+5,cy-15,cx,cy-30], fill=(200,)*3)
        elif obj == "train":
            draw.rectangle([cx-35,cy-10,cx+35,cy+10], fill=(180,50,50), outline=(0,0,0))
            for ox in [-25,-10,5,20]:
                draw.ellipse([cx+ox,cy+6,cx+ox+10,cy+16], fill=(30,)*3)
        elif obj == "traffic light":
            draw.rectangle([cx-6,cy-22,cx+6,cy+22], fill=(40,)*3)
            for i, col in enumerate([(255,50,50),(255,200,0),(50,200,50)]):
                draw.ellipse([cx-5,cy-18+i*12,cx+5,cy-8+i*12], fill=col)
        elif obj in ("fire hydrant",):
            draw.rectangle([cx-10,cy-10,cx+10,cy+10], fill=(200,50,30), outline=(0,0,0))
            draw.ellipse([cx-6,cy-14,cx+6,cy-10], fill=(200,50,30))
        elif obj == "stop sign":
            r = 18
            pts = [(cx+r*np.cos(a),cy+r*np.sin(a)) for a in np.linspace(0,2*np.pi,9)[:-1]]
            draw.polygon([(x,y) for x,y in pts], fill=(220,30,30), outline=(0,0,0))
        elif obj == "crosswalk":
            for i in range(5):
                y = cy-16+i*8
                draw.rectangle([cx-20,y,cx+20,y+4], fill=(255,255,255))
        elif obj == "bridge":
            draw.arc([cx-25,cy-20,cx+25,cy+20], 180, 360, fill=(120,)*3, width=3)
            draw.line([cx-25,cy,cx+25,cy], fill=(120,)*3, width=2)
        elif obj == "stairs":
            for i in range(5):
                draw.rectangle([cx-15+i*3,cy-10+i*5,cx+15,cy-6+i*5], fill=(150,)*3)
        elif obj in ("cat",):
            draw.ellipse([cx-12,cy-8,cx+12,cy+14], fill=(180,140,100))
            for ox in [-10,2]:
                draw.polygon([cx+ox-4,cy, cx+ox-8,cy-16, cx+ox,cy-2], fill=(180,140,100))
            draw.ellipse([cx-7,cy-3,cx-2,cy+2], fill=(30,200,30))
            draw.ellipse([cx+2,cy-3,cx+7,cy+2], fill=(30,200,30))
        elif obj in ("dog",):
            draw.ellipse([cx-12,cy-8,cx+12,cy+14], fill=(160,120,80))
            draw.ellipse([cx-18,cy-10,cx-8,cy+5], fill=(120,80,40))
            draw.ellipse([cx+8,cy-10,cx+18,cy+5], fill=(120,80,40))
            draw.ellipse([cx-5,cy-3,cx,cy+2], fill=(0,)*3)
            draw.ellipse([cx,cy-3,cx+5,cy+2], fill=(0,)*3)
        elif obj in ("bird",):
            draw.ellipse([cx-8,cy-5,cx+8,cy+5], fill=(200,150,50))
            draw.polygon([cx+8,cy-2,cx+15,cy-10,cx+12,cy], fill=(255,200,50))
        elif obj in ("palm tree",):
            draw.rectangle([cx-3,cy-5,cx+3,cy+20], fill=(100,70,30))
            for a in [-40,-15,10,35]:
                r = 20
                ex = cx+r*np.cos(np.radians(a))
                ey = cy-8+r*np.sin(np.radians(a))
                draw.ellipse([ex-12,ey-5,ex+12,ey+5], fill=(50,180,50))
        else:
            draw.ellipse([cx-15,cy-15,cx+15,cy+15], fill=(100,150,200))

    def make_tile(self, obj: str, positive: bool):
        """Return (PIL.Image, 'yes'|'no')"""
        img = Image.new("RGB", (128,128), color=(random.randint(30,220),)*3)
        draw = ImageDraw.Draw(img)
        self._bg(draw, img)
        if positive:
            self._draw(draw, obj)
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0,0.4)))
        return img, "yes" if positive else "no"

    def build(self, samples_per=250):
        dataset = []
        for obj in self.OBJECTS:
            print(f"  {obj}...", end=" ")
            for i in range(samples_per):
                img, lab = self.make_tile(obj, True)
                p = str(self.out / f"{obj}_p_{i:04d}.png")
                img.save(p)
                dataset.append({"image": p, "object": obj, "label": lab})

                other = random.choice([o for o in self.OBJECTS if o != obj])
                img, lab = self.make_tile(other, True)
                p = str(self.out / f"{obj}_n_{i:04d}.png")
                img.save(p)
                dataset.append({"image": p, "object": obj, "label": "no"})
            print("ok")

        with open(self.out / "labels.json", "w") as f:
            json.dump(dataset, f)
        print(f"\n✅ {len(dataset)} labeled tiles ({samples_per*2} × {len(self.OBJECTS)})")
        return dataset

synth = HCaptchaSynth(DATA / "synthetic")
synth_dataset = synth.build(samples_per=250)

In [ ]:
# ═══════════════════════════════════════
# 6. TRAIN Qwen2.5-VL-3B
# ═══════════════════════════════════════
# VISION model — sees images, not text.
# This is the difference between a solver
# that actually works vs one that can't see.

from unsloth import FastVisionModel
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

gc.collect()
torch.cuda.empty_cache()

# Build huggingface dataset
train_records = []
for s in synth_dataset:
    train_records.append({
        "messages": [
            {"role": "user", "content": [
                {"type": "image", "image": s["image"]},
                {"type": "text", "text": f"Is there a {s['object']} in this image? Answer ONLY 'yes' or 'no'."}
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": s["label"]}]}
        ]
    })

hf_ds = Dataset.from_list(train_records).train_test_split(test_size=0.1, seed=42)
print(f"Train: {len(hf_ds['train'])}, Test: {len(hf_ds['test'])}")

# Load model
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2.5-VL-3B-Instruct",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    r=16, lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)

print(f"✅ Model ready — {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M trainable params")

In [ ]:
# ═══════════════════════════════════════
# 7. TRAIN (≈30 min on T4)
# ═══════════════════════════════════════

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=hf_ds["train"],
    eval_dataset=hf_ds["test"],
    args=SFTConfig(
        max_seq_length=1024,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=10,
        fp16=True,
        logging_steps=25,
        eval_steps=99999,
        save_steps=99999,
        optim="adamw_8bit",
        seed=42,
        output_dir="output/hcaptcha",
        report_to="none",
        remove_unused_columns=False,
        dataset_kwargs={"skip_prepare_dataset": True},
    ),
)

print("🔥 Training...")
trainer.train()
print("✅ Done!")

In [ ]:
# ═══════════════════════════════════════
# 8. TEST THE MODEL
# ═══════════════════════════════════════

FastVisionModel.for_inference(model)

# Test: ask about a bus that IS there
test_pos, _ = synth.make_tile("bus", True)
msgs = [{"role":"user","content":[
    {"type":"image","image":test_pos},
    {"type":"text","text":"Is there a bus in this image? Answer ONLY 'yes' or 'no'."}
]}]
r = model.chat(tokenizer, msgs, max_new_tokens=5, temperature=0.1)
print(f"Bus tile (should be YES): {r}")

# Test: ask about a bus in a CAT image
test_neg, _ = synth.make_tile("cat", True)
msgs2 = [{"role":"user","content":[
    {"type":"image","image":test_neg},
    {"type":"text","text":"Is there a bus in this image? Answer ONLY 'yes' or 'no'."}
]}]
r2 = model.chat(tokenizer, msgs2, max_new_tokens=5, temperature=0.1)
print(f"Cat tile (should be NO):  {r2}")

In [ ]:
# ═══════════════════════════════════════
# 9. EXPORT GGUF + DOWNLOAD
# ═══════════════════════════════════════

model.save_pretrained_merged("output/hcaptcha/merged", tokenizer, save_method="merged_16bit")
model.save_pretrained_gguf("output/hcaptcha/gguf", tokenizer, quantization_method="q4_k_m")

!ls -lh output/hcaptcha/gguf/
!zip -r /content/hcaptcha-solver.zip output/hcaptcha/ -q

from google.colab import files
files.download("/content/hcaptcha-solver.zip")
print("\n✅ GGUF exported + download started")

## 🎯 What you just built

| Step | What | Result |
|------|------|--------|
| **Scrape** | 100 hCaptcha challenges from `accounts.hcaptcha.com/demo` | Real challenge screenshots |
| **Extract** | 3×3 tiles from each screenshot | 900 real hCaptcha tiles |
| **Synthetic** | 19 objects × 500 tiles each | 9,500 labeled tiles |
| **Train** | Qwen2.5-VL-3B on the labeled data | Model that says yes/no to "is X in this image?" |
| **Export** | GGUF Q4_K_M | Ready for `ollama create` |

### How the scraper works
```
for _ in range(100):
    open hcaptcha.com/demo
    type "1" in the field
    click submit
    wait for hCaptcha to appear
    screenshot the challenge
    close browser
    wait 5 seconds
    repeat
```

### To actually solve a full hCaptcha
Once the model classifies tiles, the solver pipeline is:
```python
# 1. Extract prompt text ("Select all images with BUSES")
# 2. Extract the 9 tiles
# 3. Ask model "Is there a BUS in this tile?" for each tile
# 4. Click tiles where answer is "yes"
# 5. Click "Verify" button
```